In [12]:
import importlib
import model.spat_only_lora
importlib.reload(model.spat_only_lora)
from model.spat_only_lora import SpatOnlyLoRA

In [13]:

import os
import sys
import torch
import torch.nn as nn
import numpy as np
import scipy.io as sio
from torch.utils.data import TensorDataset, DataLoader
import warnings
warnings.filterwarnings("ignore")

notebook_dir = os.path.dirname(os.path.abspath("__file__" if "__file__" in globals() else "."))
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# %% [markdown]
# ## 1. 超参数设置

# %%
patch_size = 8
img_size = 128
pca_components = 20
class_num = 16  # Indian Pines has 16 classes

lora_rank = 8
lora_alpha = 1.0
train_num = 10

data_path = "data/Indian_pines_corrected.mat"
gt_path = "data/Indian_pines_gt.mat"
os.makedirs("result", exist_ok=True)

# %% [markdown]
# ## 2. 数据加载与预处理

from model import split_data, utils, create_graph

raw_data = sio.loadmat(data_path)["data"].astype(np.float32)
gt = sio.loadmat(gt_path)["groundT"].astype(np.float32)
height_orgin, width_orgin, bands = raw_data.shape

data, _ = split_data.apply_PCA(raw_data, num_components=pca_components)
print(f"Data shape after PCA: {data.shape}")

gt_reshape = gt.reshape(-1)
train_index = []
test_index = []

for cls in range(1, class_num + 1):
    cls_pixels = np.where(gt_reshape == cls)[0]
    if len(cls_pixels) == 0:
        continue
    np.random.seed(42)
    np.random.shuffle(cls_pixels)
    train_index.extend(cls_pixels[:train_num])
    test_index.extend(cls_pixels[train_num:])

train_index = np.array(train_index, dtype=int)
test_index = np.array(test_index, dtype=int)
val_index = np.array([], dtype=int)

print(f"Train samples: {len(train_index)}, Test samples: {len(test_index)}")

train_samples_gt, test_samples_gt, _ = create_graph.get_label(gt_reshape, train_index, val_index, test_index)
train_gt_onehot = create_graph.label_to_one_hot(train_samples_gt.reshape(height_orgin, width_orgin), class_num)
test_gt_onehot = create_graph.label_to_one_hot(test_samples_gt.reshape(height_orgin, width_orgin), class_num)

train_samples_gt = torch.from_numpy(train_samples_gt.astype(np.float32)).to(device)
test_samples_gt = torch.from_numpy(test_samples_gt.astype(np.float32)).to(device)

# 获取图像块用于推理（切成 num_H × num_W 个 128x128 块）
img_train, num_H, num_W, data_gt, data = utils.Get_train_and_test_data(img_size, data, gt)
height, width, _ = data.shape
img_train = torch.from_numpy(img_train.transpose(0, 3, 1, 2)).float()
train_dataset = TensorDataset(img_train)
train_loader = DataLoader(train_dataset, batch_size=num_H, shuffle=False)

# %% [markdown]
# ## 3. 模型定义：SpatViT + LoRA（ViT Dense Prediction 风格）

# %%
from model.spat_only_lora import SpatOnlyLoRA

model = SpatOnlyLoRA(
    img_size=img_size,
    in_channels=pca_components,
    patch_size=patch_size,
    classes=class_num,
    model_size='base',
    lora_rank=lora_rank,
    lora_alpha=lora_alpha
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params / 1e6:.2f}M | Trainable: {trainable_params / 1e6:.2f}M ({trainable_params/total_params*100:.2f}%)")

# %% [markdown]
# ## 4. 加载预训练权重（移除不兼容层）

# %%
spat_net = torch.load("spat-base.pth", map_location='cpu')
spat_weights = spat_net['model']

keys_to_remove = [
    'patch_embed.proj',      # 输入通道不同
    'pos_embed',             # 尺寸可能不匹配
    'cls_token',             # ViT 分类用，我们做 dense prediction 不需要
    'head',                  # ⚠️ 分类头必须删！
    'spat_map',
    'spat_output_maps'
]

for k in list(spat_weights.keys()):
    if any(rm in k for rm in keys_to_remove):
        del spat_weights[k]

# 添加 encoder 前缀
spat_weights = {'encoder.' + k: v for k, v in spat_weights.items()}

model.load_state_dict(spat_weights, strict=False)
print("✅ Pretrained weights loaded!")

# %% [markdown]
# ## 5. 推理（ViT Dense Prediction 风格）

# %%
model.eval()
with torch.no_grad():
    # pred 存储每个 128x128 块的预测结果 [num_W, num_H, C, H, W]
    pred = torch.zeros(num_W, num_H, class_num, img_size, img_size, device=device)

    for batch_idx, (batch_data,) in enumerate(train_loader):
        # batch_data: [num_H, C, 128, 128]
        for i in range(num_H):
            netinput = batch_data[i].unsqueeze(0).to(device)  # [1, C, 128, 128]
            
            # ✅ 关键：模型应直接输出 [1, class_num, 128, 128]
            batch_pred = model(netinput)  # [1, class_num, 128, 128]
            
            # 直接 squeeze 第一维
            pred[batch_idx, i] = batch_pred.squeeze(0)  # [class_num, 128, 128]

    # 重组为全图 [H_total, W_total, class_num]
    pred = pred.permute(2, 1, 3, 0, 4).reshape(class_num, -1).permute(1, 0)  # [H*W, class_num]
    y_orgin = utils.image_reshape(pred, height, width, height_orgin, width_orgin, class_num)

# %% [markdown]
# ## 6. 性能评估

# %%
zeros = torch.zeros(height_orgin * width_orgin).to(device).float()
overall_acc, _, average_acc, kappa, each_acc = utils.evaluate_performance_all(
    y_orgin, test_samples_gt, test_gt_onehot,
    height_orgin, width_orgin, class_num, gt,
    device, require_AA_KPP=True, printFlag=True
)

# %% [markdown]
# ## 7. 可视化

# %%
import matplotlib.pyplot as plt

pred_map = torch.argmax(y_orgin, dim=1).cpu().numpy().reshape(height_orgin, width_orgin)
gt_map = gt

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(gt_map, cmap='nipy_spectral')
plt.title("Ground Truth")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(pred_map, cmap='nipy_spectral')
plt.title(f"Prediction (OA: {overall_acc:.4f})")
plt.axis('off')

plt.tight_layout()
plt.savefig("result/spat_lora_prediction.png", dpi=300, bbox_inches='tight')
plt.show()

Using device: cuda:0


Data shape after PCA: (149, 149, 20)
Train samples: 160, Test samples: 10089
padding img: (256, 256, 20)
Total params: 94.03M | Trainable: 72.76M (77.39%)
✅ Pretrained weights loaded!


IndexError: list index out of range